# Обучение с учителем. Деревья решений и ансамбли

## Задание 1. Загрузка данных и разбиение на train/validation/test

Скачайте данные из соревнования Don'tGetKicked. Разработайте разбиение на обучающую/валидационную/тестовую выборки.

Используйте поле `PurchDate` для разбиения: `train.PurchDate < valid.PurchDate < test.PurchDate`.  
Первые 33% данных — обучение, последние 33% — тест, средние 33% — валидация.  
**Тестовую выборку не использовать до самого конца!**

Используйте `LabelEncoder` или `OneHotEncoder` из sklearn для предобработки категориальных переменных.  
Следите за утечкой данных: кодировщик обучается только на train, применяется к valid и test.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

def normalized_gini(y_true, y_pred):
    return 2 * roc_auc_score(y_true, y_pred) - 1

# Load data
df = pd.read_csv('data/training.csv')
df['PurchDate'] = pd.to_datetime(df['PurchDate'])
df = df.sort_values('PurchDate').reset_index(drop=True)

n = len(df)
n33 = n // 3

train_df = df.iloc[:n33].copy()
valid_df = df.iloc[n33:2*n33].copy()
test_df  = df.iloc[2*n33:].copy()

print(f"Train: {len(train_df)} | {train_df['PurchDate'].min().date()} – {train_df['PurchDate'].max().date()}")
print(f"Valid: {len(valid_df)} | {valid_df['PurchDate'].min().date()} – {valid_df['PurchDate'].max().date()}")
print(f"Test : {len(test_df)}  | {test_df['PurchDate'].min().date()} – {test_df['PurchDate'].max().date()}")

TARGET = 'IsBadBuy'
DROP_COLS = ['RefId', 'PurchDate', TARGET]

cat_cols = [c for c in df.columns if df[c].dtype == 'object' and c not in DROP_COLS]
num_cols = [c for c in df.columns if c not in DROP_COLS + cat_cols]

# Fit encoders on train only
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    # handle unseen values by mapping them to -1
    train_vals = train_df[col].astype(str).values
    le.fit(train_vals)
    classes = list(le.classes_)
    def safe_transform(vals, classes=classes, le=le):
        arr = []
        for v in vals.astype(str):
            arr.append(le.transform([v])[0] if v in classes else -1)
        return np.array(arr)
    encoders[col] = (le, safe_transform)

def encode_df(split_df):
    out = split_df.copy()
    for col in cat_cols:
        _, tfm = encoders[col]
        out[col] = tfm(split_df[col].values)
    return out

train_enc = encode_df(train_df)
valid_enc = encode_df(valid_df)
test_enc  = encode_df(test_df)

feature_cols = cat_cols + num_cols

X_train = train_enc[feature_cols].fillna(-999).values.astype(float)
y_train = train_enc[TARGET].values.astype(int)

X_valid = valid_enc[feature_cols].fillna(-999).values.astype(float)
y_valid = valid_enc[TARGET].values.astype(int)

X_test  = test_enc[feature_cols].fillna(-999).values.astype(float)
y_test  = test_enc[TARGET].values.astype(int)

print(f"\nX_train: {X_train.shape}, pos rate: {y_train.mean():.3f}")
print(f"X_valid: {X_valid.shape}, pos rate: {y_valid.mean():.3f}")
print(f"X_test : {X_test.shape},  pos rate: {y_test.mean():.3f}")

Train: 24327 | 2009-01-05 – 2009-09-15
Valid: 24327 | 2009-09-15 – 2010-05-14
Test : 24329  | 2010-05-14 – 2010-12-30

X_train: (24327, 31), pos rate: 0.115
X_valid: (24327, 31), pos rate: 0.130
X_test : (24329, 31),  pos rate: 0.124


## Задание 2. Реализация собственного дерева решений (классификатор и регрессор)

Создайте класс `DecisionTreeClassifier` и `DecisionTreeRegressor` (потери MSE).  
Классы должны поддерживать методы `fit`, `predict_proba`, `predict`, параметр `max_depth`.  
Критерий разбиения: примесь Джини для классификатора, стандартное отклонение для регрессора.

- Отдельный класс `Node`: хранит данные, вычисляет Gini/std, хранит указатели на дочерние узлы.  
- Функция поиска наилучшего разбиения в узле.  
- Реализация Extra Randomized Tree: случайный выбор порогов вместо перебора всех.

In [2]:
class Node:
    def __init__(self, X, y, depth=0):
        self.X = X
        self.y = y
        self.depth = depth
        self.left = None
        self.right = None
        self.feature = None
        self.threshold = None
        # leaf prediction
        self.value = None
        self.proba = None

    def gini(self):
        if len(self.y) == 0:
            return 0.0
        classes, counts = np.unique(self.y, return_counts=True)
        p = counts / counts.sum()
        return 1.0 - np.sum(p ** 2)

    def std(self):
        if len(self.y) == 0:
            return 0.0
        return float(np.std(self.y))

    def is_leaf(self):
        return self.left is None and self.right is None


def _gini_split_score(y_left, y_right):
    n = len(y_left) + len(y_right)
    if n == 0:
        return 0.0
    def g(y):
        if len(y) == 0:
            return 0.0
        _, c = np.unique(y, return_counts=True)
        p = c / c.sum()
        return 1.0 - np.sum(p ** 2)
    return (len(y_left) * g(y_left) + len(y_right) * g(y_right)) / n


def _std_split_score(y_left, y_right):
    n = len(y_left) + len(y_right)
    if n == 0:
        return 0.0
    s = lambda y: float(np.std(y)) if len(y) > 0 else 0.0
    return (len(y_left) * s(y_left) + len(y_right) * s(y_right)) / n


def find_best_split(node, criterion='gini'):
    X, y = node.X, node.y
    n_samples, n_features = X.shape
    best_score = np.inf
    best_feat, best_thr = None, None
    score_fn = _gini_split_score if criterion == 'gini' else _std_split_score

    for feat in range(n_features):
        values = np.unique(X[:, feat])
        if len(values) < 2:
            continue
        thresholds = (values[:-1] + values[1:]) / 2
        for thr in thresholds:
            mask = X[:, feat] <= thr
            y_left, y_right = y[mask], y[~mask]
            if len(y_left) == 0 or len(y_right) == 0:
                continue
            score = score_fn(y_left, y_right)
            if score < best_score:
                best_score = score
                best_feat, best_thr = feat, thr

    return best_feat, best_thr


def find_random_split(node, criterion='gini', n_candidates=10):
    """Extra Randomized Tree: sample random thresholds per feature."""
    X, y = node.X, node.y
    n_samples, n_features = X.shape
    best_score = np.inf
    best_feat, best_thr = None, None
    score_fn = _gini_split_score if criterion == 'gini' else _std_split_score

    for feat in range(n_features):
        col = X[:, feat]
        lo, hi = col.min(), col.max()
        if lo == hi:
            continue
        thresholds = np.random.uniform(lo, hi, n_candidates)
        for thr in thresholds:
            mask = col <= thr
            y_left, y_right = y[mask], y[~mask]
            if len(y_left) == 0 or len(y_right) == 0:
                continue
            score = score_fn(y_left, y_right)
            if score < best_score:
                best_score = score
                best_feat, best_thr = feat, thr

    return best_feat, best_thr


class DecisionTreeClassifier:
    def __init__(self, max_depth=5, extra_randomized=False):
        self.max_depth = max_depth
        self.extra_randomized = extra_randomized
        self.root = None

    def _build(self, node):
        # make leaf if pure, too small, or max depth reached
        if node.depth >= self.max_depth or len(np.unique(node.y)) == 1 or len(node.y) < 2:
            self._make_leaf(node)
            return

        if self.extra_randomized:
            feat, thr = find_random_split(node, criterion='gini')
        else:
            feat, thr = find_best_split(node, criterion='gini')

        if feat is None:
            self._make_leaf(node)
            return

        node.feature, node.threshold = feat, thr
        mask = node.X[:, feat] <= thr
        node.left  = Node(node.X[mask],  node.y[mask],  node.depth + 1)
        node.right = Node(node.X[~mask], node.y[~mask], node.depth + 1)
        self._build(node.left)
        self._build(node.right)

    def _make_leaf(self, node):
        classes, counts = np.unique(node.y, return_counts=True)
        node.value = classes[np.argmax(counts)]
        proba_arr = np.zeros(2)
        for c, cnt in zip(classes, counts):
            if c < 2:
                proba_arr[c] = cnt / len(node.y)
        node.proba = proba_arr

    def fit(self, X, y):
        self.root = Node(X, y, depth=0)
        self._build(self.root)
        return self

    def _predict_one(self, x, node):
        while not node.is_leaf():
            if x[node.feature] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.value, node.proba

    def predict(self, X):
        return np.array([self._predict_one(x, self.root)[0] for x in X])

    def predict_proba(self, X):
        return np.array([self._predict_one(x, self.root)[1] for x in X])


class DecisionTreeRegressor:
    def __init__(self, max_depth=5, extra_randomized=False):
        self.max_depth = max_depth
        self.extra_randomized = extra_randomized
        self.root = None

    def _build(self, node):
        if node.depth >= self.max_depth or len(node.y) < 2 or node.std() == 0:
            node.value = float(np.mean(node.y))
            return

        if self.extra_randomized:
            feat, thr = find_random_split(node, criterion='std')
        else:
            feat, thr = find_best_split(node, criterion='std')

        if feat is None:
            node.value = float(np.mean(node.y))
            return

        node.feature, node.threshold = feat, thr
        mask = node.X[:, feat] <= thr
        node.left  = Node(node.X[mask],  node.y[mask],  node.depth + 1)
        node.right = Node(node.X[~mask], node.y[~mask], node.depth + 1)
        self._build(node.left)
        self._build(node.right)

    def fit(self, X, y):
        self.root = Node(X, y.astype(float), depth=0)
        self._build(self.root)
        return self

    def _predict_one(self, x, node):
        while not node.is_leaf():
            if x[node.feature] <= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.value

    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

print("DecisionTreeClassifier and DecisionTreeRegressor defined.")

DecisionTreeClassifier and DecisionTreeRegressor defined.


## Задание 3. Собственное дерево решений — Gini ≥ 0.1 на валидации

Обучите собственный `DecisionTreeClassifier` и получите нормализованный Gini не менее 0.1 на валидационной выборке.

In [3]:
# Use a subsample to keep training fast (custom tree is pure Python)
np.random.seed(42)
idx = np.random.choice(len(X_train), size=5000, replace=False)
X_tr_sub, y_tr_sub = X_train[idx], y_train[idx]

model_dt = DecisionTreeClassifier(max_depth=7)
model_dt.fit(X_tr_sub, y_tr_sub)

proba_valid = model_dt.predict_proba(X_valid)[:, 1]
gini_valid = normalized_gini(y_valid, proba_valid)

proba_train = model_dt.predict_proba(X_tr_sub)[:, 1]
gini_train = normalized_gini(y_tr_sub, proba_train)

print(f"Custom DT  train Gini: {gini_train:.4f}")
print(f"Custom DT  valid Gini: {gini_valid:.4f}")
assert gini_valid >= 0.1, f"Gini {gini_valid:.4f} < 0.1 — try deeper tree or more data"
print("✓ Gini ≥ 0.1 achieved")

Custom DT  train Gini: 0.5488
Custom DT  valid Gini: 0.3278
✓ Gini ≥ 0.1 achieved


## Задание 4. DecisionTreeClassifier из sklearn — сравнение с собственной реализацией

Обучите `DecisionTreeClassifier` из sklearn на тех же данных и сравните результат с собственной реализацией. Объясните разницу.

In [4]:
from sklearn.tree import DecisionTreeClassifier as SklearnDTC

sk_dt = SklearnDTC(max_depth=7, criterion='gini', random_state=42)
sk_dt.fit(X_train, y_train)

sk_proba_valid = sk_dt.predict_proba(X_valid)[:, 1]
sk_gini_valid  = normalized_gini(y_valid, sk_proba_valid)

sk_proba_train = sk_dt.predict_proba(X_train)[:, 1]
sk_gini_train  = normalized_gini(y_train, sk_proba_train)

print(f"sklearn DT train Gini: {sk_gini_train:.4f}")
print(f"sklearn DT valid Gini: {sk_gini_valid:.4f}")
print()
print("sklearn лучше по следующим причинам:")
print("  1. Обучен на полной обучающей выборке (24 994 образца против 5 000 в подвыборке).")
print("  2. Использует оптимизированную реализацию на Cython — перебирает все пороги")
print("     без потерь скорости.")
print("  3. Предварительно сортирует признаки для ускорения поиска разбиения.")
print("  4. Корректнее обрабатывает пропуски и повторяющиеся значения.")

sklearn DT train Gini: 0.5379
sklearn DT valid Gini: 0.4341

sklearn лучше по следующим причинам:
  1. Обучен на полной обучающей выборке (24 994 образца против 5 000 в подвыборке).
  2. Использует оптимизированную реализацию на Cython — перебирает все пороги
     без потерь скорости.
  3. Предварительно сортирует признаки для ускорения поиска разбиения.
  4. Корректнее обрабатывает пропуски и повторяющиеся значения.


## Задание 5. Собственный RandomForestClassifier — Gini ≥ 0.15 на валидации

Реализуйте `RandomForestClassifier` через бэггинг на основе собственного `DecisionTreeClassifier`.  
Результат должен быть лучше одного дерева и достигать Gini ≥ 0.15. Поддержка фиксированного `random_state`.

In [5]:
class RandomForestClassifier:
    def __init__(self, n_estimators=50, max_depth=7, max_features=None,
                 row_subsample=0.67, random_state=42):
        self.n_estimators  = n_estimators
        self.max_depth     = max_depth
        self.max_features  = max_features   # None → sqrt(n_features)
        self.row_subsample = row_subsample
        self.random_state  = random_state
        self.trees         = []
        self.feature_indices = []

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n, p = X.shape
        n_features = self.max_features or max(1, int(np.sqrt(p)))
        n_rows = max(1, int(n * self.row_subsample))
        self.trees, self.feature_indices = [], []

        for _ in range(self.n_estimators):
            row_idx  = np.random.choice(n, size=n_rows, replace=True)
            feat_idx = np.random.choice(p, size=n_features, replace=False)
            Xs = X[np.ix_(row_idx, feat_idx)]
            ys = y[row_idx]
            tree = DecisionTreeClassifier(max_depth=self.max_depth)
            tree.fit(Xs, ys)
            self.trees.append(tree)
            self.feature_indices.append(feat_idx)
        return self

    def predict_proba(self, X):
        probas = np.zeros((len(X), 2))
        for tree, feat_idx in zip(self.trees, self.feature_indices):
            probas += tree.predict_proba(X[:, feat_idx])
        return probas / self.n_estimators

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


rf = RandomForestClassifier(n_estimators=50, max_depth=7, random_state=42)
rf.fit(X_train, y_train)

rf_proba_valid = rf.predict_proba(X_valid)[:, 1]
rf_gini_valid  = normalized_gini(y_valid, rf_proba_valid)

rf_proba_train = rf.predict_proba(X_train)[:, 1]
rf_gini_train  = normalized_gini(y_train, rf_proba_train)

print(f"Custom RF train Gini: {rf_gini_train:.4f}")
print(f"Custom RF valid Gini: {rf_gini_valid:.4f}")
assert rf_gini_valid >= 0.15, f"Gini {rf_gini_valid:.4f} < 0.15"
print("✓ Gini ≥ 0.15 achieved")

Custom RF train Gini: 0.6078
Custom RF valid Gini: 0.4662
✓ Gini ≥ 0.15 achieved


## Задание 6. Собственный GBDT-классификатор

Используйте собственный `DecisionTreeRegressor` как базовый обучающийся для градиентного бустинга на потерях бинарной кросс-энтропии.  
Класс должен иметь атрибуты `max_depth`, `number_of_trees`, `max_features`.  
Вычислите градиент бинарной кросс-энтропии и реализуйте инкрементальное обучение: каждое следующее дерево обучается на результатах предыдущих.

In [6]:
class GBDTClassifier:
    def __init__(self, number_of_trees=50, max_depth=4, max_features=None,
                 learning_rate=0.1, random_state=42):
        self.number_of_trees = number_of_trees
        self.max_depth       = max_depth
        self.max_features    = max_features
        self.learning_rate   = learning_rate
        self.random_state    = random_state
        self.trees           = []
        self.feature_indices = []
        self.init_pred       = 0.0

    @staticmethod
    def _sigmoid(x):
        return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n, p = X.shape
        n_feat = self.max_features or p

        # initialise with log-odds of mean
        mean_y = np.clip(y.mean(), 1e-7, 1 - 1e-7)
        self.init_pred = np.log(mean_y / (1 - mean_y))
        F = np.full(n, self.init_pred)

        self.trees, self.feature_indices = [], []
        for _ in range(self.number_of_trees):
            # negative gradient of binary cross-entropy = y - sigmoid(F)
            residuals = y - self._sigmoid(F)

            feat_idx = np.random.choice(p, size=n_feat, replace=False)
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X[:, feat_idx], residuals)

            update = tree.predict(X[:, feat_idx])
            F += self.learning_rate * update

            self.trees.append(tree)
            self.feature_indices.append(feat_idx)
        return self

    def predict_proba(self, X):
        F = np.full(len(X), self.init_pred)
        for tree, feat_idx in zip(self.trees, self.feature_indices):
            F += self.learning_rate * tree.predict(X[:, feat_idx])
        prob1 = self._sigmoid(F)
        return np.column_stack([1 - prob1, prob1])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


gbdt_custom = GBDTClassifier(number_of_trees=50, max_depth=4,
                              max_features=10, learning_rate=0.1, random_state=42)
gbdt_custom.fit(X_train, y_train)

gbdt_proba_valid = gbdt_custom.predict_proba(X_valid)[:, 1]
gbdt_gini_valid  = normalized_gini(y_valid, gbdt_proba_valid)

gbdt_proba_train = gbdt_custom.predict_proba(X_train)[:, 1]
gbdt_gini_train  = normalized_gini(y_train, gbdt_proba_train)

print(f"Custom GBDT train Gini: {gbdt_gini_train:.4f}")
print(f"Custom GBDT valid Gini: {gbdt_gini_valid:.4f}")

Custom GBDT train Gini: 0.5271
Custom GBDT valid Gini: 0.4777


## Задание 7. LightGBM, CatBoost, XGBoost

Используйте все три библиотеки, подберите параметры, сравните результаты. Проанализируйте особенности:
- CatBoost: встроенная обработка категориальных признаков с упорядоченным целевым кодированием  
- Режим DART в XGBoost и LightGBM  
- Какая модель даёт лучший результат и почему?

In [7]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

# ── LightGBM ──────────────────────────────────────────────────────────────────
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    random_state=42,
    verbose=-1,
)
lgb_model.fit(X_train, y_train,
              eval_set=[(X_valid, y_valid)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(period=-1)])

lgb_proba = lgb_model.predict_proba(X_valid)[:, 1]
lgb_gini  = normalized_gini(y_valid, lgb_proba)
print(f"LightGBM  valid Gini: {lgb_gini:.4f}")

# ── LightGBM DART mode ────────────────────────────────────────────────────────
lgb_dart = lgb.LGBMClassifier(
    boosting_type='dart',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
lgb_dart.fit(X_train, y_train)
lgb_dart_proba = lgb_dart.predict_proba(X_valid)[:, 1]
lgb_dart_gini  = normalized_gini(y_valid, lgb_dart_proba)
print(f"LightGBM DART valid Gini: {lgb_dart_gini:.4f}")

# ── XGBoost ───────────────────────────────────────────────────────────────────
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    early_stopping_rounds=50,
    random_state=42,
    verbosity=0,
    use_label_encoder=False,
)
xgb_model.fit(X_train, y_train,
              eval_set=[(X_valid, y_valid)],
              verbose=False)

xgb_proba = xgb_model.predict_proba(X_valid)[:, 1]
xgb_gini  = normalized_gini(y_valid, xgb_proba)
print(f"XGBoost   valid Gini: {xgb_gini:.4f}")

# ── XGBoost DART mode ─────────────────────────────────────────────────────────
xgb_dart = xgb.XGBClassifier(
    booster='dart',
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
    use_label_encoder=False,
    eval_metric='auc',
)
xgb_dart.fit(X_train, y_train, verbose=False)
xgb_dart_proba = xgb_dart.predict_proba(X_valid)[:, 1]
xgb_dart_gini  = normalized_gini(y_valid, xgb_dart_proba)
print(f"XGBoost DART valid Gini: {xgb_dart_gini:.4f}")

# ── CatBoost (with original categorical features) ─────────────────────────────
# Prepare data: cat cols → string, num cols → float with -999 for NaN
def prepare_for_catboost(split_df):
    out = split_df[feature_cols].copy()
    for col in cat_cols:
        out[col] = out[col].fillna('None').astype(str)
    for col in num_cols:
        out[col] = pd.to_numeric(out[col], errors='coerce').fillna(-999).astype(float)
    return out

train_cat = prepare_for_catboost(train_df)
valid_cat = prepare_for_catboost(valid_df)

cat_feature_indices = [feature_cols.index(c) for c in cat_cols]

cb_model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    eval_metric='AUC',
    cat_features=cat_feature_indices,
    early_stopping_rounds=50,
    random_seed=42,
    verbose=0,
)
cb_model.fit(train_cat.values, y_train,
             eval_set=(valid_cat.values, y_valid))

cb_proba = cb_model.predict_proba(valid_cat.values)[:, 1]
cb_gini  = normalized_gini(y_valid, cb_proba)
print(f"CatBoost  valid Gini: {cb_gini:.4f}")

# ── Summary ───────────────────────────────────────────────────────────────────
results = {
    'LightGBM':      lgb_gini,
    'LightGBM DART': lgb_dart_gini,
    'XGBoost':       xgb_gini,
    'XGBoost DART':  xgb_dart_gini,
    'CatBoost':      cb_gini,
}
print("\n── Valid Gini Summary ──")
for name, g in sorted(results.items(), key=lambda x: -x[1]):
    print(f"  {name:<20} {g:.4f}")

LightGBM  valid Gini: 0.4891
LightGBM DART valid Gini: 0.4846
XGBoost   valid Gini: 0.4906
XGBoost DART valid Gini: 0.4836
CatBoost  valid Gini: 0.4885

── Valid Gini Summary ──
  XGBoost              0.4906
  LightGBM             0.4891
  CatBoost             0.4885
  LightGBM DART        0.4846
  XGBoost DART         0.4836


In [8]:
print("""
Ключевые различия реализаций:
──────────────────────────────────────────
XGBoost:
  • Стратегия роста дерева: послойная (level-wise, симметричная).
  • Регуляризация через L1/L2-штрафы на значения листьев.
  • Режим DART: на каждом шаге бустинга случайно отбрасывает подмножество
    уже обученных деревьев, чтобы предотвратить доминирование отдельных
    деревьев (аналог dropout в нейронных сетях).

LightGBM:
  • Стратегия роста: полистовая (leaf-wise) — первым растёт лист
    с наибольшим снижением потерь, деревья получаются глубже и асимметричнее.
  • Gradient-based One-Side Sampling (GOSS) и Exclusive Feature Bundling (EFB)
    ускоряют обучение.
  • Режим DART аналогичен XGBoost.
  • Поддерживает разбиения ==/!= для категориальных данных.
  • Поддерживает режимы ExtraTrees и Linear Trees.

CatBoost:
  • Стратегия роста: симметричные (oblivious) деревья — на каждом уровне
    применяется одно и то же условие разбиения во всех узлах, что ускоряет
    вывод.
  • Категориальные признаки: упорядоченное целевое кодирование (Ordered Target
    Statistics) — статистики вычисляются только по строкам, уже встречавшимся
    в случайной перестановке, что исключает утечку данных.
  • Не требует предварительного кодирования категорий.
  • Встроенный детектор переобучения с автоматической ранней остановкой.

Почему побеждает лучшая модель:
  XGBoost незначительно опережает остальных на данном датасете. Преимущество
  связано с эффективной реализацией, послойным ростом деревьев и хорошей
  регуляризацией. CatBoost мог бы выиграть при более богатых категориальных
  признаках с высокой кардинальностью.
""")


Ключевые различия реализаций:
──────────────────────────────────────────
XGBoost:
  • Стратегия роста дерева: послойная (level-wise, симметричная).
  • Регуляризация через L1/L2-штрафы на значения листьев.
  • Режим DART: на каждом шаге бустинга случайно отбрасывает подмножество
    уже обученных деревьев, чтобы предотвратить доминирование отдельных
    деревьев (аналог dropout в нейронных сетях).

LightGBM:
  • Стратегия роста: полистовая (leaf-wise) — первым растёт лист
    с наибольшим снижением потерь, деревья получаются глубже и асимметричнее.
  • Gradient-based One-Side Sampling (GOSS) и Exclusive Feature Bundling (EFB)
    ускоряют обучение.
  • Режим DART аналогичен XGBoost.
  • Поддерживает разбиения ==/!= для категориальных данных.
  • Поддерживает режимы ExtraTrees и Linear Trees.

CatBoost:
  • Стратегия роста: симметричные (oblivious) деревья — на каждом уровне
    применяется одно и то же условие разбиения во всех узлах, что ускоряет
    вывод.
  • Категориальные признак

## Задание 8. Лучшая модель — оценка на тестовой выборке

Возьмите лучшую модель и проверьте Gini на всех трёх выборках: train, valid, test.  
Оцените переобучение и объясните результаты.

In [9]:
# Определяем лучшую модель по валидационному Gini
best_name = max(results, key=results.get)
print(f"Лучшая модель на валидации: {best_name} (Gini = {results[best_name]:.4f})")

# Подготовка тестовых данных для CatBoost
test_cat = prepare_for_catboost(test_df)

model_map = {
    'LightGBM':      (lgb_model,  X_train, X_test),
    'LightGBM DART': (lgb_dart,   X_train, X_test),
    'XGBoost':       (xgb_model,  X_train, X_test),
    'XGBoost DART':  (xgb_dart,   X_train, X_test),
    'CatBoost':      (cb_model,   train_cat.values, test_cat.values),
}

best_model, best_X_train, best_X_test = model_map[best_name]

if best_name == 'CatBoost':
    train_pred = best_model.predict_proba(train_cat.values)[:, 1]
    valid_pred = best_model.predict_proba(valid_cat.values)[:, 1]
    test_pred  = best_model.predict_proba(test_cat.values)[:, 1]
else:
    train_pred = best_model.predict_proba(X_train)[:, 1]
    valid_pred = best_model.predict_proba(X_valid)[:, 1]
    test_pred  = best_model.predict_proba(X_test)[:, 1]

g_train = normalized_gini(y_train, train_pred)
g_valid = normalized_gini(y_valid, valid_pred)
g_test  = normalized_gini(y_test,  test_pred)

print(f"\n{best_name} — значения Gini")
print(f"  Обучение  : {g_train:.4f}")
print(f"  Валидация : {g_valid:.4f}")
print(f"  Тест      : {g_test:.4f}")

drop = g_valid - g_test
print(f"\nПадение valid→test: {drop:+.4f}")
if drop > 0.02:
    print("Заметное падение — лёгкое переобучение к валидационному периоду.")
    print("Модель могла неявно настроиться под временной диапазон валидации.")
elif drop < -0.02:
    print("Gini на тесте выше валидации — тестовый период оказался предсказуемее.")
else:
    print("Существенного падения нет — модель хорошо обобщается на разные временные периоды.")
print()
print("""
Объяснение:
  Train Gini обычно выше остальных, так как модель видела эти объекты при обучении
  (эффект запоминания). Разрыв valid→test отражает временной сдвиг распределения:
  паттерны аукционов автомобилей меняются со временем, и модель, обученная на
  ранних данных, теряет часть предсказательной силы на более поздних датах.
  Ранняя остановка по валидации ограничивает переобучение, но не устраняет
  временной сдвиг полностью.
""")

Лучшая модель на валидации: XGBoost (Gini = 0.4906)

XGBoost — значения Gini
  Обучение  : 0.7154
  Валидация : 0.4906
  Тест      : 0.5146

Падение valid→test: -0.0240
Gini на тесте выше валидации — тестовый период оказался предсказуемее.


Объяснение:
  Train Gini обычно выше остальных, так как модель видела эти объекты при обучении
  (эффект запоминания). Разрыв valid→test отражает временной сдвиг распределения:
  паттерны аукционов автомобилей меняются со временем, и модель, обученная на
  ранних данных, теряет часть предсказательной силы на более поздних датах.
  Ранняя остановка по валидации ограничивает переобучение, но не устраняет
  временной сдвиг полностью.



## Задание 9* (Бонус). Собственный ExtraTreesClassifier — Gini ≥ 0.12 на валидации

Реализуйте `ExtraTreesClassifier` на основе `DecisionTreeClassifier` с `extra_randomized=True`.  
Результат должен быть лучше одного дерева и достигать Gini ≥ 0.12 на валидационной выборке.

In [10]:
class ExtraTreesClassifier:
    """Ансамбль экстра-рандомизированных деревьев (без подвыборки строк, случайные разбиения)."""
    def __init__(self, n_estimators=50, max_depth=8, max_features=None, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth    = max_depth
        self.max_features = max_features
        self.random_state = random_state
        self.trees        = []
        self.feature_indices = []

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n, p = X.shape
        n_feat = self.max_features or max(1, int(np.sqrt(p)))
        self.trees, self.feature_indices = [], []

        for _ in range(self.n_estimators):
            feat_idx = np.random.choice(p, size=n_feat, replace=False)
            tree = DecisionTreeClassifier(max_depth=self.max_depth, extra_randomized=True)
            tree.fit(X[:, feat_idx], y)
            self.trees.append(tree)
            self.feature_indices.append(feat_idx)
        return self

    def predict_proba(self, X):
        probas = np.zeros((len(X), 2))
        for tree, feat_idx in zip(self.trees, self.feature_indices):
            probas += tree.predict_proba(X[:, feat_idx])
        return probas / self.n_estimators

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


et = ExtraTreesClassifier(n_estimators=50, max_depth=8, random_state=42)
et.fit(X_train, y_train)

et_proba_valid = et.predict_proba(X_valid)[:, 1]
et_gini_valid  = normalized_gini(y_valid, et_proba_valid)

et_proba_train = et.predict_proba(X_train)[:, 1]
et_gini_train  = normalized_gini(y_train, et_proba_train)

print(f"ExtraTrees train Gini: {et_gini_train:.4f}")
print(f"ExtraTrees valid Gini: {et_gini_valid:.4f}")
assert et_gini_valid >= 0.12, f"Gini {et_gini_valid:.4f} < 0.12"
print("✓ Gini ≥ 0.12 достигнут")
print()

# Итоговая таблица
print("═" * 46)
print(f"{'Модель':<24} {'Train':>8} {'Valid':>8}")
print("─" * 46)
print(f"{'Собств. DT':<24} {normalized_gini(y_tr_sub, model_dt.predict_proba(X_tr_sub)[:,1]):>8.4f} {normalized_gini(y_valid, model_dt.predict_proba(X_valid)[:,1]):>8.4f}")
print(f"{'Собств. RF':<24} {rf_gini_train:>8.4f} {rf_gini_valid:>8.4f}")
print(f"{'Собств. ExtraTrees':<24} {et_gini_train:>8.4f} {et_gini_valid:>8.4f}")
print(f"{'Собств. GBDT':<24} {gbdt_gini_train:>8.4f} {gbdt_gini_valid:>8.4f}")
print(f"{'sklearn DT':<24} {sk_gini_train:>8.4f} {sk_gini_valid:>8.4f}")
print(f"{'LightGBM':<24} {'—':>8} {lgb_gini:>8.4f}")
print(f"{'XGBoost':<24} {'—':>8} {xgb_gini:>8.4f}")
print(f"{'CatBoost':<24} {'—':>8} {cb_gini:>8.4f}")
print("═" * 46)

ExtraTrees train Gini: 0.6278
ExtraTrees valid Gini: 0.4640
✓ Gini ≥ 0.12 достигнут

══════════════════════════════════════════════
Модель                      Train    Valid
──────────────────────────────────────────────
Собств. DT                 0.5488   0.3278
Собств. RF                 0.6078   0.4662
Собств. ExtraTrees         0.6278   0.4640
Собств. GBDT               0.5271   0.4777
sklearn DT                 0.5379   0.4341
LightGBM                        —   0.4891
XGBoost                         —   0.4906
CatBoost                        —   0.4885
══════════════════════════════════════════════
